In [39]:
!apt-get update && apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [52]:
!mkdir -p sequence_detector
!cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0  19996      0 --:--:-- --:--:-- --:--:-- 20133


In [53]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?
'''

In [56]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=""
)

completion = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"role":"user","content":verilog_generation_prompt}],
    max_tokens=1024,
    stream=False
)

print(completion.choices[0].message.content)

Here's a Verilog model for the sequence detector that meets the specified requirements:

```verilog
module sequence_detector(
    input  wire        clk,
    input  wire        rst_n,
    input  wire  [2:0] data,
    output wire        seq_found
);

reg [3:0] sequence_state;
reg [7:0] sequence_state_history;

always @(posedge clk) begin
    if (!rst_n) begin
        sequence_state <= 0;
        sequence_state_history <= 0;
    end else begin
        // Store the current data in the sequence history
        sequence_state_history[7:0] <= {sequence_state_history[5:0], data};
        
        // Check if the sequence is found
        if (sequence_state_history[7:0] == 8'b110110110) begin
            // Check if the sequence is found in the last 8 bits
            if (sequence_state_history[7:0] == 8'b110110110) begin
                seq_found <= 1;
                // Reset the sequence state
                sequence_state <= 0;
            end else begin
                // Reset the seque

In [61]:
output_verilog_code = '''
module sequence_detector (
    input         clk,
    input         reset_n,
    input  [2:0]  data,
    output        sequence_found
);

    reg [3:0] state;

    assign sequence_found = (state == 4'd7 && data == 3'b101);

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= 4'd0;
        end else begin
            case (state)
                4'd0: state <= (data == 3'b001) ? 4'd1 : 4'd0;
                4'd1: state <= (data == 3'b101) ? 4'd2 : 4'd0;
                4'd2: state <= (data == 3'b110) ? 4'd3 : 4'd0;
                4'd3: state <= (data == 3'b000) ? 4'd4 : 4'd0;
                4'd4: state <= (data == 3'b110) ? 4'd5 : 4'd0;
                4'd5: state <= (data == 3'b110) ? 4'd6 : 4'd0;
                4'd6: state <= (data == 3'b011) ? 4'd7 : 4'd0;
                4'd7: state <= 4'd0;
                default: state <= 4'd0;
            endcase
        end
    end

endmodule
'''

filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(output_verilog_code)

!cd sequence_detector/ && iverilog -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

All test cases passed.


Debugging Loop – Iteration Log

This section documents the iterative improvement process for the sequence detector. I adjusted the prompt/code based on compile errors and failing tests across at least 3 iterations.

Iteration 1: The LLM produced Verilog with wrong port names (rst_n and seq_found instead of reset_n and sequence_found) and incorrect logic. It checked for an 8-bit sequence (8'b110110110) instead of detecting the exact 8-value 3-bit sequence 001, 101, 110, 000, 110, 110, 011, 101. I replaced that with an 8-state FSM: S_IDLE, S_001, S_101, S_110, S_000, S_110_2, S_110_3, S_011. After running iverilog and vvp, the output was: Error: Cycle 8, Expected: 1, Got: 0.

Iteration 2: The design compiled, but iverilog reported "This assignment requires an explicit cast" around lines 31–37. This was caused by using typedef enum and assigning to it in the combinational always @(*) block. I removed the enum and used localparam plus reg [3:0] for the state and next_state. After recompiling and running vvp, the output was again: Error: Cycle 8, Expected: 1, Got: 0.

Iteration 3: The testbench expected sequence_found to be 1 at cycle 8, but it was driven in the sequential always @(posedge clk) block. Because of non-blocking assignments, the testbench sampled sequence_found before it was updated. I changed sequence_found to a combinational output with assign sequence_found = (state == 4'd7 && data == 3'b101); so it updates as soon as state and data are correct. After rerunning iverilog and vvp, all tests passed.

Final command used: iverilog -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp